# 04 — Spark SQL & `notebookutils` (mssparkutils)

Covers: temp/global views, DDL patterns, and the Fabric utility library (`notebookutils`,
formerly `mssparkutils`) used for file operations, secrets, notebook orchestration and
parameterization — the "glue" that turns standalone notebooks into a pipeline.


## 1. Temp views vs global temp views

In [ ]:
df = spark.read.table("silver_patient_visits")

# Session-scoped — only visible to cells in THIS notebook session
df.createOrReplaceTempView("visits_view")

# Visible across notebooks attached to the same Spark application (e.g. in a pipeline
# that runs several notebooks in the same session pool)
df.createOrReplaceGlobalTempView("visits_global_view")


In [ ]:
%%sql
SELECT department, COUNT(*) AS visit_count
FROM visits_view
GROUP BY department
ORDER BY visit_count DESC


In [ ]:
%%sql
SELECT * FROM global_temp.visits_global_view LIMIT 5


## 2. DDL patterns you'll actually use

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS gold_dept_daily_visits (
    department STRING,
    visit_date DATE,
    visit_count BIGINT,
    updated_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (visit_date);

CREATE OR REPLACE VIEW vw_high_risk_visits AS
SELECT * FROM silver_patient_visits WHERE age >= 65;

-- Shallow/deep clone for creating a dev copy without duplicating storage
CREATE TABLE dev_accounts_clone SHALLOW CLONE accounts;


## 3. `notebookutils.fs` — file system operations

This is the Fabric-native replacement for Databricks' `dbutils.fs`, addressing OneLake paths.

In [ ]:
from notebookutils import mssparkutils   # also importable directly as `notebookutils`

# List files
mssparkutils.fs.ls("Files/raw/patient_visits")

# Create a directory
mssparkutils.fs.mkdirs("Files/archive/2026-01")

# Copy / move a processed batch into an archive folder
mssparkutils.fs.cp("Files/raw/patient_visits/2026-01", "Files/archive/2026-01", recurse=True)
mssparkutils.fs.mv("Files/raw/patient_visits/2026-01", "Files/processed/2026-01", create_path=True)

# Remove old staging files after a successful load
mssparkutils.fs.rm("Files/staging/tmp_batch", recurse=True)

# Read a small file directly (e.g. a control/config file) without Spark
content = mssparkutils.fs.head("Files/config/pipeline_control.json", maxBytes=2000)
print(content)


## 4. Secrets via Azure Key Vault

In [ ]:
from notebookutils import mssparkutils

sql_password = mssparkutils.credentials.getSecret(
    "https://<your-keyvault-name>.vault.azure.net/", "sql-connection-password"
)
# Never print secrets — pass them straight into a connection string / JDBC options dict.


## 5. Notebook parameters & orchestration

Mark a cell's **Toggle parameter cell** setting (or add `# PARAMETERS CELL` as a tag) so a
pipeline's *Run Notebook* activity can inject values at run time.

In [ ]:
# This is the "parameters" cell — tag it as a parameter cell in the notebook UI
run_date = "2026-01-15"
source_folder = "Files/raw/patient_visits"
env = "dev"


In [ ]:
from notebookutils import mssparkutils

# Orchestrate: call a child notebook and pass it parameters; the call blocks until it finishes
result = mssparkutils.notebook.run(
    "03_Data_Ingestion_and_Transformation",
    timeout_seconds=3600,
    arguments={"run_date": run_date, "source_folder": source_folder},
)
print("Child notebook result:", result)


In [ ]:
# Run several notebooks concurrently (e.g. independent Silver builds) and wait for all
dag = {
    "activities": [
        {"name": "silver_visits", "path": "03_Data_Ingestion_and_Transformation",
         "timeoutPerCellInSeconds": 600, "args": {"run_date": run_date}},
        {"name": "gold_dept_summary", "path": "07_Real_World_Medallion_ETL_Banking",
         "timeoutPerCellInSeconds": 900, "args": {"run_date": run_date}},
    ]
}
mssparkutils.notebook.runMultiple(dag, {"concurrency": 2})


In [ ]:
from notebookutils import mssparkutils

# Exit a notebook early and pass a value back to the caller (pipeline or parent notebook)
if run_date is None:
    mssparkutils.notebook.exit("FAILED: run_date parameter missing")
else:
    mssparkutils.notebook.exit(f"SUCCESS: processed batch for {run_date}")


## 6. Discovering Lakehouse/workspace metadata at runtime

In [ ]:
from notebookutils import mssparkutils

# List lakehouses in the current workspace
lakehouses = mssparkutils.lakehouse.list()
for lh in lakehouses:
    print(lh["displayName"], lh["id"])

# Current runtime context — handy for logging which workspace/notebook/run this is
ctx = mssparkutils.runtime.context
print(ctx)


Next notebook: **05 — Performance Tuning & Optimization**.